In [1]:
import pandas as pd
import numpy as np
import torch
from imblearn.over_sampling import SMOTE

from pathlib import Path

In [2]:
class Data():
    def __init__(self, tcga_project:str, tcga_dir:str='./../../datasets/tcga/'):
        gene_counts_dir = Path(tcga_dir) / f'{tcga_project}_gene_counts.csv'
        self.gene_counts = pd.read_csv(gene_counts_dir)

        metadata_dir = Path(tcga_dir) / f'{tcga_project}_metadata.csv'
        self.metadata = pd.read_csv(metadata_dir)



In [3]:
data = Data(
    tcga_project='TCGA-BRCA'
)

In [4]:
data.gene_counts

,Unnamed: 0,TCGA-A2-A25D-01A-12R-A16F-07,TCGA-BH-A201-01A-11R-A14M-07,TCGA-AC-A23C-01A-12R-A169-07,TCGA-AR-A5QP-01A-11R-A28M-07,TCGA-C8-A12P-01A-11R-A115-07,TCGA-BH-A0W3-01A-11R-A109-07,TCGA-AC-A8OQ-01A-11R-A41B-07,TCGA-BH-A0H6-01A-21R-A056-07,TCGA-D8-A27V-01A-12R-A17B-07,...,TCGA-AC-A3TM-01A-11R-A22K-07,TCGA-A7-A26J-01A-11R-A277-07,TCGA-LL-A441-01A-11R-A24H-07,TCGA-A7-A13G-01A-11R-A13Q-07,TCGA-BH-A0E6-01A-11R-A034-07,TCGA-LL-A5YP-01A-21R-A28M-07,TCGA-AO-A03L-01A-41R-A056-07,TCGA-BH-A42T-01A-11R-A24H-07,TCGA-A2-A04W-01A-31R-A115-07,TCGA-AR-A24O-01A-11R-A169-07
0,ENSG00000000003.15,4474,3555,731,3960,3834,826,690,5534,13509,...,1153,599,5172,3299,7389,2300,955,2641,2718,1856
1,ENSG00000000005.6,9,30,113,50,0,8,55,29,15,...,5,0,68,24,20,7,3,40,1,58
2,ENSG00000000419.13,1785,2537,5947,2155,3161,1483,2456,1511,3060,...,1118,2843,4333,1465,1366,2524,3231,2311,1914,2184
3,ENSG00000000457.14,1362,1865,3118,1355,1411,1594,646,929,7099,...,573,2677,2584,5199,650,720,1763,1263,677,1206
4,ENSG00000000460.17,433,890,1063,474,637,645,764,300,1449,...,281,930,971,1257,1414,632,899,904,291,574
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60655,ENSG00000288669.1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
60656,ENSG00000288670.1,150,590,754,332,354,238,347,333,1240,...,184,328,395,459,407,140,416,561,121,287
60657,ENSG00000288671.1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
60658,ENSG00000288674.1,6,11,14,9,0,11,13,7,8,...,7,9,35,12,14,6,9,57,1,9


In [5]:
data.metadata

,Unnamed: 0,barcode,patient,sample,shortLetterCode,definition,sample_submitter_id,sample_type_id,tumor_descriptor,sample_id,...,paper_tobacco_smoking_history,paper_CNV.Clusters,paper_Mutation.Clusters,paper_DNA.Methylation.Clusters,paper_mRNA.Clusters,paper_miRNA.Clusters,paper_lncRNA.Clusters,paper_Protein.Clusters,paper_PARADIGM.Clusters,paper_Pan.Gyn.Clusters
0,TCGA-A2-A25D-01A-12R-A16F-07,TCGA-A2-A25D-01A-12R-A16F-07,TCGA-A2-A25D,TCGA-A2-A25D-01A,TP,Primary solid Tumor,TCGA-A2-A25D-01A,1,Primary,eda0ff5c-84cf-44cb-888f-0aab17441dfa,...,NaN,C6,C4,C1,C2,C2,C2,C1,C4,C3
1,TCGA-BH-A201-01A-11R-A14M-07,TCGA-BH-A201-01A-11R-A14M-07,TCGA-BH-A201,TCGA-BH-A201-01A,TP,Primary solid Tumor,TCGA-BH-A201-01A,1,Primary,7893162b-3d19-4c54-86ee-aa4884d84e8a,...,NaN,C5,C4,C1,C2,C3,C2,C1,C6,C1
2,TCGA-AC-A23C-01A-12R-A169-07,TCGA-AC-A23C-01A-12R-A169-07,TCGA-AC-A23C,TCGA-AC-A23C-01A,TP,Primary solid Tumor,TCGA-AC-A23C-01A,1,Primary,9cbd4bd6-244b-4274-a230-a7bd02e79571,...,NaN,C3,C1,C1,C1,C3,C1,C1,C6,C3
3,TCGA-AR-A5QP-01A-11R-A28M-07,TCGA-AR-A5QP-01A-11R-A28M-07,TCGA-AR-A5QP,TCGA-AR-A5QP-01A,TP,Primary solid Tumor,TCGA-AR-A5QP-01A,1,Primary,151a1187-e07e-4b76-b2d2-2da133b9116b,...,NaN,C5,C7,C1,C2,C2,NaN,C2,C6,C1
4,TCGA-C8-A12P-01A-11R-A115-07,TCGA-C8-A12P-01A-11R-A115-07,TCGA-C8-A12P,TCGA-C8-A12P-01A,TP,Primary solid Tumor,TCGA-C8-A12P-01A,1,Primary,0d5aa7a3-3488-4339-b8f1-e4971912e2bc,...,NaN,C4,C9,C2,C2,C3,C1,C5,C4,C4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1226,TCGA-LL-A5YP-01A-21R-A28M-07,TCGA-LL-A5YP-01A-21R-A28M-07,TCGA-LL-A5YP,TCGA-LL-A5YP-01A,TP,Primary solid Tumor,TCGA-LL-A5YP-01A,1,Primary,3dc0bc4f-6863-4297-8915-f29e1aa25d26,...,NaN,C4,C1,C4,C4,C7,NaN,C5,C4,C3
1227,TCGA-AO-A03L-01A-41R-A056-07,TCGA-AO-A03L-01A-41R-A056-07,TCGA-AO-A03L,TCGA-AO-A03L-01A,TP,Primary solid Tumor,TCGA-AO-A03L-01A,1,Primary,00f81e98-e5c3-43b2-9757-f769ec17f3f8,...,NaN,C6,C1,C2,C2,C3,C1,C1,C5,C4
1228,TCGA-BH-A42T-01A-11R-A24H-07,TCGA-BH-A42T-01A-11R-A24H-07,TCGA-BH-A42T,TCGA-BH-A42T-01A,TP,Primary solid Tumor,TCGA-BH-A42T-01A,1,Primary,312a82ee-9236-4282-b5eb-ee6063babf5a,...,NaN,C6,C4,C2,C2,C2,NaN,C1,C8,C4
1229,TCGA-A2-A04W-01A-31R-A115-07,TCGA-A2-A04W-01A-31R-A115-07,TCGA-A2-A04W,TCGA-A2-A04W-01A,TP,Primary solid Tumor,TCGA-A2-A04W-01A,1,Primary,699d9e14-0968-4773-b077-17db941f5c38,...,NaN,C4,C9,C4,C1,C3,C1,C5,C5,C4
